# [Feature Update (2026-08-16)] Feature policy rules (General availability) — 検証ノートブック

## このノートブックについて

Zenn 記事「[[Feature Update (2026-08-16)] Feature policy rules (General availability)](https://zenn.dev/inoway/articles/i268-feature-update-2026-08-16-feature-policy-rule)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## セットアップ

事前準備（DB・スキーマ・ロール・権限）


In [ ]:
CREATE DATABASE IF NOT EXISTS VERIFY_TEMP_DB;
CREATE SCHEMA IF NOT EXISTS VERIFY_TEMP_DB.POLICY_TEST;

CREATE ROLE IF NOT EXISTS VERIFY_ADMIN_ROLE;
CREATE ROLE IF NOT EXISTS VERIFY_ANALYST_ROLE;

GRANT USAGE ON DATABASE VERIFY_TEMP_DB TO ROLE VERIFY_ADMIN_ROLE;
GRANT USAGE ON DATABASE VERIFY_TEMP_DB TO ROLE VERIFY_ANALYST_ROLE;
GRANT USAGE ON SCHEMA VERIFY_TEMP_DB.POLICY_TEST TO ROLE VERIFY_ADMIN_ROLE;
GRANT USAGE ON SCHEMA VERIFY_TEMP_DB.POLICY_TEST TO ROLE VERIFY_ANALYST_ROLE;

GRANT CREATE TABLE ON SCHEMA VERIFY_TEMP_DB.POLICY_TEST TO ROLE VERIFY_ADMIN_ROLE;
GRANT CREATE TABLE ON SCHEMA VERIFY_TEMP_DB.POLICY_TEST TO ROLE VERIFY_ANALYST_ROLE;

-- feature policy の作成にはこの権限が必要です
GRANT CREATE FEATURE POLICY ON SCHEMA VERIFY_TEMP_DB.POLICY_TEST
  TO ROLE VERIFY_ADMIN_ROLE;

GRANT ROLE VERIFY_ADMIN_ROLE TO USER DATA_USER;
GRANT ROLE VERIFY_ANALYST_ROLE TO USER DATA_USER;

## ステップ1: ロール×IP のポリシーを作る

条件式には端末を表す IP アドレスを直接書きます。
本記事では `203.0.113.10` を社内の開発用 PC、`198.51.100.20` を出先のノート PC として扱います。
検証では、実際の接続元 IP を開発用 PC の IP とみなして実行しています。
記事中の IP はドキュメント用に予約されたアドレス（RFC 5737）に置き換えています。


In [ ]:
-- 管理ロールは開発用 PC 以外からブロック
-- 分析ロールは開発用 PC からブロック
CREATE OR REPLACE FEATURE POLICY VERIFY_TEMP_DB.POLICY_TEST.COMBINED_ROLE_IP_POLICY
  AS $$
    conditions:
      - name: admin_from_wrong_terminal
        expression: "CURRENT_ROLE() = 'VERIFY_ADMIN_ROLE' AND CURRENT_IP_ADDRESS() <> '203.0.113.10'"
      - name: analyst_from_wrong_terminal
        expression: "CURRENT_ROLE() = 'VERIFY_ANALYST_ROLE' AND CURRENT_IP_ADDRESS() = '203.0.113.10'"
    blocked_creation_rules:
      - object_type: TABLE
        block_when_any:
          - admin_from_wrong_terminal
          - analyst_from_wrong_terminal
  $$;

## ステップ2: DESCRIBE で YAML ボディを確認する


In [ ]:
-- policy_definition プロパティに YAML ボディが入る
DESCRIBE FEATURE POLICY VERIFY_TEMP_DB.POLICY_TEST.COMBINED_ROLE_IP_POLICY;

## ステップ3: データベースにポリシーを適用する


In [ ]:
-- スキーマ単位ではなくデータベース単位で紐づける
ALTER DATABASE VERIFY_TEMP_DB
  SET FEATURE POLICY VERIFY_TEMP_DB.POLICY_TEST.COMBINED_ROLE_IP_POLICY;

## ステップ4: 許可される組み合わせを試す

管理ロール（`VERIFY_ADMIN_ROLE`）で、開発用 PC から作成します。


In [ ]:
USE ROLE VERIFY_ADMIN_ROLE;
CREATE TABLE VERIFY_TEMP_DB.POLICY_TEST.ADMIN_TABLE_FROM_DEV (id INT);

## ステップ5: ブロックされる組み合わせを試す

同じ端末のまま、分析ロールに切り替えて作成します。


In [ ]:
USE ROLE VERIFY_ANALYST_ROLE;
CREATE TABLE VERIFY_TEMP_DB.POLICY_TEST.ANALYST_TABLE_FROM_DEV (id INT);

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。

In [ ]:
DROP SCHEMA IF EXISTS VERIFY_TEMP_DB.POLICY_TEST;
DROP TABLE IF EXISTS ON;
DROP TABLE IF EXISTS VERIFY_TEMP_DB.POLICY_TEST.ADMIN_TABLE_FROM_DEV;
DROP TABLE IF EXISTS VERIFY_TEMP_DB.POLICY_TEST.ANALYST_TABLE_FROM_DEV;

-- データベースを削除（内包するオブジェクトもすべて削除）
DROP DATABASE IF EXISTS VERIFY_TEMP_DB;